# Processing of a resting-state EEG dataset for sleep deprivation

Bad channels are interpolated

## PRE PROCESSING

#### The intention is to provide epochs(4 seconds) to the machine and try predicting if the case is sleep deprivation or normal sleep.

In [1]:
from matplotlib import pyplot as plt
import os
import pandas as pd
import numpy as np
from scipy import signal
import mne
import mne_icalabel
from mne.preprocessing import ICA
from autoreject import Ransac  
from mne_icalabel import label_components
%matplotlib inline

In [2]:
# lists to collect data
X_list = []
y_list = []
data_NS_list =[]
data_SD_list =[]

In [3]:
n_epochs = 0
for i in range(1, 72):
    if i == 28 or i== 1 or i== 44 or i==39 or i==43 or i in [7,8,11,13,18,24,29,31,35,36,37,38,40,47,48,50,54,58,59,63,67,70,71]: #46 participants with complete data
        continue  # Skip participants due to missing data
    for j in range(1, 3):
        
        #Define file path
        file_path = f'D:\\BCI\\TASKS\\Task-8\\ds004902_data\\sub-{i:02d}\\ses-{j}\\eeg\\sub-{i:02d}_ses-{j}_task-eyesopen_eeg.set'#The d indicates that the value is a decimal integer, while the 02 specifies that the output should be padded with leading zeros if necessary to reach a width of two characters.
        epochs_file = f'D:\\BCI\\TASKS\\Task-8\\processed_epochs_60\\sub-{i:02d}_ses-{j}_epochs.fif'  # Path to save/load processed epochs
        
        
        if os.path.exists(epochs_file):
            print(f"Loading processed epochs from {epochs_file}")
            epochs = mne.read_epochs(epochs_file, preload=True)
        else:
            print(f"\nProcessing: {file_path}\n")

            #Define EEG channels
            eeg_channels = ["Fp1", "AF3", "AF7", "Fz", "F1", "F3", "F5", "F7","FC1", "FC3", "FC5", "FT7","Cz", "C1", "C3", "C5", "T7","CP1", "CP3", "CP5", "TP7", "TP9","Pz", "P1", "P3", "P5", "P7","PO3", "PO7", "Oz", "O1","Fpz", "Fp2", "AF4", "AF8","F2", "F4", "F6", "F8","FC2", "FC4", "FC6", "FT8","C2", "C4", "C6", "T8","CPz", "CP2", "CP4", "CP6","TP8", "TP10","P2", "P4", "P6", "P8","POz", "PO4", "PO8", "O2"]
            
            #Load EEG data
            raw = mne.io.read_raw_eeglab(file_path, preload=True)
            eeg_raw=raw.pick(mne.pick_channels(raw.info['ch_names'], include=eeg_channels))
            
            #Montage setting
            montage = mne.channels.make_standard_montage('standard_1020')
            eeg_raw.set_montage(montage)

            #Filtering
            eeg_raw.filter(1., 40.)#, fir_design='firwin')

            #Average Referencing
            eeg_raw.set_eeg_reference('average')

            #EPOCHING
            epochs = mne.make_fixed_length_epochs(eeg_raw, duration=60.0, overlap=0.0,preload=True)

            #RANSAC for bad channel detection
            ransac = Ransac(verbose=False, n_jobs=1)
            ransac.fit(epochs)
            epochs.info['bads'].extend(ransac.bad_chs_)   

            #interpolate bad channels
            epochs.interpolate_bads(reset_bads=True)
            
            #ICA for artifact removal
            ica = mne.preprocessing.ICA(n_components=20, method='infomax', fit_params=dict(extended=True), random_state=42, max_iter=1000)
            ica.fit(epochs)

            ic_labels = label_components(epochs, ica, method='iclabel')
            exclude_idx = []
            for idx, label in enumerate(ic_labels["labels"]):
                if label not in ["brain","other","muscle artifact"] and ic_labels["y_pred_proba"][idx] > 0.5:
                    exclude_idx.append(idx)

            ica.apply(epochs,exclude= exclude_idx)           

            # Save processed epochs
            os.makedirs(os.path.dirname(epochs_file), exist_ok=True) #creates the necessary directories for the file path if they don't exist.
            epochs.save(epochs_file, overwrite=True)
            print(f"Saved processed epochs to {epochs_file}")

        if j==1:
            labels = 0
        elif j==2:
            labels = 1
        labels = pd.DataFrame({'label': [labels]*len(epochs)})
        
        # Extract data and labels
        data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)

        #if (i == 39 or i == 43) and j == 2:
        #    data = signal.decimate(data, 10, axis=2) # Downsample by a factor of 10 to match time points.

        epoch_labels = labels['label'].values  
        
        if i <= 71:  
            X_list.append(data)
            y_list.append(epoch_labels)

        if j==1:
            data_NS_list.append(data)
        elif j==2:
            data_SD_list.append(data)

# After the loop, concatenate into arrays
X= np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
data_SD = np.concatenate(data_SD_list, axis=0)
data_NS = np.concatenate(data_NS_list, axis=0)

Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-1_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-1_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available
Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated


C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\3940900364.py:14: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-1_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-2_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-2_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available
Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated


C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\3940900364.py:14: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-2_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-03_ses-1_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-03_ses-1_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available
Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated


...

Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated
Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-1_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-1_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available


C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\3940900364.py:14: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-1_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated
Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-2_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-2_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available


C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\3940900364.py:14: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-2_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated


In [4]:
print(X.shape)
print(y.shape)
print(data_SD.shape)
print(data_NS.shape)
print(X)
print(y)

(424, 61, 30000)
(424,)
(210, 61, 30000)
(214, 61, 30000)
[[[-1.86939514e-09 -1.33587992e-06 -2.67785286e-06 ...  3.39845633e-06
    2.90296911e-06  2.38385891e-06]
  [-1.94064725e-09  1.98257356e-07  1.31205871e-07 ...  6.59083508e-06
    6.43760393e-06  6.10524176e-06]
  [-1.08360790e-09 -6.29877148e-07 -1.39925456e-06 ...  2.80599474e-06
    2.63128828e-06  2.36371660e-06]
  ...
  [ 4.23254502e-10  1.06595909e-06  2.29082608e-06 ...  1.41882562e-06
    2.76843046e-06  3.73536276e-06]
  [ 4.71726466e-10  2.87471890e-07  8.53711305e-07 ...  1.00523967e-05
    1.02179270e-05  9.79988763e-06]
  [ 5.23308931e-10  1.85923862e-06  3.72308849e-06 ...  3.99468016e-06
    6.16059874e-06  7.70940350e-06]]

 [[ 1.86271786e-06  1.33896160e-06  7.96520708e-07 ... -9.06401095e-07
   -8.24688671e-08  6.52927635e-07]
  [ 5.61213015e-06  4.99401473e-06  4.29776095e-06 ... -4.90488957e-06
   -4.45498608e-06 -4.03521632e-06]
  [ 2.02848935e-06  1.65482366e-06  1.27060425e-06 ... -2.20463204e-06
   -1.5

## Balancing the no of epochs

In [5]:
min_length = min(len(data_NS), len(data_SD))
print("Minimum length for balancing:", min_length)
def balance_data(data_array, target_length):
    # If this is the larger array, sample it down
    if len(data_array) > target_length:
        # random_state=42 ensures you get the same 'random' choice every time (reproducible)
        indices = np.random.choice(len(data_array), target_length, replace=False)
        return data_array[indices]
    return data_array

Minimum length for balancing: 210


In [6]:
data_NS = balance_data(data_NS, min_length)
data_SD = balance_data(data_SD, min_length)
print("Balanced data_NS shape:", data_NS.shape)
print("Balanced data_SD shape:", data_SD.shape)

Balanced data_NS shape: (210, 61, 30000)
Balanced data_SD shape: (210, 61, 30000)


In [7]:
#no of epoches obtained
n_epochs = data_NS.shape[0] + data_SD.shape[0]
print("Total number of epochs:", n_epochs)

Total number of epochs: 420


## Band Power Extraction

In [8]:
# Bandpower extraction functions
def bandpower_delta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    delta_band = (0.5, 4)
    mask = (freqs >= delta_band[0]) & (freqs <= delta_band[1])
    delta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return delta_power

def bandpower_theta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    theta_band = (4, 8)
    mask = (freqs >= theta_band[0]) & (freqs <= theta_band[1])
    theta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return theta_power

def bandpower_alpha(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    alpha_band = (8, 13)
    mask = (freqs >= alpha_band[0]) & (freqs <= alpha_band[1])
    alpha_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return alpha_power  

def bandpower_beta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    beta_band = (13, 30)
    mask = (freqs >= beta_band[0]) & (freqs <= beta_band[1])
    beta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return beta_power       

def bandpower_gamma(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    gamma_band = (30, 40)
    mask = (freqs >= gamma_band[0]) & (freqs <= gamma_band[1])
    gamma_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return gamma_power

def bandpower_total(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    total_band = (0.5, 40)
    mask = (freqs >= total_band[0]) & (freqs <= total_band[1])
    total_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return total_power


In [9]:
# Average Bandpower extraction functions
def average_bandpower_delta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    delta_band = (0.5, 4)
    mask = (freqs >= delta_band[0]) & (freqs <= delta_band[1])
    delta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis) # dimensions: (n_epochs, n_channels)
    average_delta_bandpower = np.mean(delta_power, axis= -1)  # Average across channels
    return average_delta_bandpower

def average_bandpower_theta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    theta_band = (4, 8)
    mask = (freqs >= theta_band[0]) & (freqs <= theta_band[1])
    theta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_theta_bandpower = np.mean(theta_power, axis=-1)  # Average across channels
    return average_theta_bandpower

def average_bandpower_alpha(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    alpha_band = (8, 13)
    mask = (freqs >= alpha_band[0]) & (freqs <= alpha_band[1])
    alpha_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_alpha_bandpower = np.mean(alpha_power, axis=-1)  # Average across channels
    return average_alpha_bandpower      

def average_bandpower_beta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    beta_band = (13, 30)
    mask = (freqs >= beta_band[0]) & (freqs <= beta_band[1])
    beta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_beta_bandpower = np.mean(beta_power, axis=-1)  # Average across channels
    return average_beta_bandpower

def average_bandpower_gamma(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    gamma_band = (30, 40)
    mask = (freqs >= gamma_band[0]) & (freqs <= gamma_band[1])
    gamma_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_gamma_bandpower = np.mean(gamma_power, axis=-1)  # Average across channels
    return average_gamma_bandpower

def average_bandpower_total(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    total_band = (0.5, 40)
    mask = (freqs >= total_band[0]) & (freqs <= total_band[1])
    total_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_total_bandpower = np.mean(total_power, axis=-1)  # Average across channels
    return average_total_bandpower

In [10]:
#calculation of delta band powers for both SD and NS data
from tqdm import tqdm_notebook
delta_power_SD_list=[]
for datta in tqdm_notebook(data_SD):
    ith_delta_powers_SD = bandpower_delta(datta, fs=500)
    delta_power_SD_list.append(ith_delta_powers_SD)

delta_power_SD=np.array(delta_power_SD_list)
delta_power_SD.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\133475402.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

In [11]:
#calculation of delta band powers for both SD and NS data
from tqdm import tqdm_notebook
delta_power_NS_list=[]
for datta in tqdm_notebook(data_NS):
    ith_delta_powers_NS = bandpower_delta(datta, fs=500, axis=-1)
    delta_power_NS_list.append(ith_delta_powers_NS)

delta_power_NS=np.array(delta_power_NS_list)
delta_power_NS.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\3985284175.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_NS):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

In [12]:
#calculation of theta band powers for both SD data
from tqdm import tqdm_notebook
theta_power_SD_list=[]
for datta in tqdm_notebook(data_SD):
    ith_theta_powers_SD = bandpower_theta(datta, fs=500, axis=-1)
    theta_power_SD_list.append(ith_theta_powers_SD)

theta_power_SD=np.array(theta_power_SD_list)
theta_power_SD.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\759687591.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

In [13]:
#calculation of theta band powers for both NS data
from tqdm import tqdm_notebook
theta_power_NS_list=[]
for datta in tqdm_notebook(data_NS):
    ith_theta_powers_NS = bandpower_theta(datta, fs=500, axis=-1)
    theta_power_NS_list.append(ith_theta_powers_NS)

theta_power_NS=np.array(theta_power_NS_list)
theta_power_NS.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\2564802599.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_NS):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

In [14]:
#calculation of alpha band powers for both SD data
from tqdm import tqdm_notebook
alpha_power_SD_list=[]
for datta in tqdm_notebook(data_SD):
    ith_alpha_powers_SD = bandpower_alpha(datta, fs=500, axis=-1)
    alpha_power_SD_list.append(ith_alpha_powers_SD)

alpha_power_SD=np.array(alpha_power_SD_list)
alpha_power_SD.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\705825680.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

In [15]:
#calculation of alpha band powers for both NS data
from tqdm import tqdm_notebook
alpha_power_NS_list=[]
for datta in tqdm_notebook(data_NS):
    ith_alpha_powers_NS = bandpower_alpha(datta, fs=500, axis=-1)
    alpha_power_NS_list.append(ith_alpha_powers_NS)

alpha_power_NS=np.array(alpha_power_NS_list)
alpha_power_NS.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\1609555356.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_NS):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

In [16]:
#calculation of beta band powers for both SD data
from tqdm import tqdm_notebook
beta_power_SD_list=[]
for datta in tqdm_notebook(data_SD):
    ith_beta_powers_SD = bandpower_beta(datta, fs=500, axis=-1)
    beta_power_SD_list.append(ith_beta_powers_SD)

beta_power_SD=np.array(beta_power_SD_list)
beta_power_SD.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\3865207504.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

In [17]:
#calculation of beta band powers for NS data
from tqdm import tqdm_notebook
beta_power_NS_list=[]
for datta in tqdm_notebook(data_NS):
    ith_beta_powers_NS = bandpower_beta(datta, fs=500, axis=-1)
    beta_power_NS_list.append(ith_beta_powers_NS)

beta_power_NS=np.array(beta_power_NS_list)
beta_power_NS.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\3158172285.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_NS):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

In [18]:
#calculation of gamma band powers for SD data
from tqdm import tqdm_notebook
gamma_power_SD_list=[]
for datta in tqdm_notebook(data_SD):
    ith_gamma_powers_SD = bandpower_gamma(datta, fs=500, axis=-1)
    gamma_power_SD_list.append(ith_gamma_powers_SD)

gamma_power_SD=np.array(gamma_power_SD_list)
gamma_power_SD.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\204014121.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

In [19]:
#calculation of gamma band powers for NS data
from tqdm import tqdm_notebook
gamma_power_NS_list=[]
for datta in tqdm_notebook(data_NS):
    ith_gamma_powers_NS = bandpower_gamma(datta, fs=500, axis=-1)
    gamma_power_NS_list.append(ith_gamma_powers_NS)

gamma_power_NS=np.array(gamma_power_NS_list)
gamma_power_NS.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_14688\2721333161.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(data_NS):


  0%|          | 0/210 [00:00<?, ?it/s]

(210, 61)

## Absolute Powers

In [20]:
from scipy import stats
from statsmodels.stats.weightstats import ztest

In [21]:
t_stat_delta, p_delta_t = stats.ttest_ind(delta_power_SD, delta_power_NS, equal_var=False)
z_stat_delta, p_delta_z = ztest(delta_power_SD, delta_power_NS, value=0)
print("DELTA: \n","T_stat:\n", t_stat_delta,"\n","P_stat:\n", p_delta_t,"\n Z_stat:\n", z_stat_delta,"\n p_stat:\n", p_delta_z)

import pandas as pd

eeg_channels = ["Fp1", "AF3", "AF7", "Fz", "F1", "F3", "F5", "F7","FC1", "FC3", "FC5", "FT7","Cz", "C1", "C3", "C5", "T7","CP1", "CP3", "CP5", "TP7", "TP9","Pz", "P1", "P3", "P5", "P7","PO3", "PO7", "Oz", "O1","Fpz", "Fp2", "AF4", "AF8","F2", "F4", "F6", "F8","FC2", "FC4", "FC6", "FT8","C2", "C4", "C6", "T8","CPz", "CP2", "CP4", "CP6","TP8", "TP10","P2", "P4", "P6", "P8","POz", "PO4", "PO8", "O2"]
# Create the DataFrame
df_delta = pd.DataFrame({
    'Channel': eeg_channels,
    'T_Stat': t_stat_delta,
    'P_Value': p_delta_t,
    'Significance': np.where(p_delta_t < 0.05, 'Yes', 'No') 
})
print(df_delta['Significance'].value_counts())
# Filter the DataFrame for significant channels
significant_channels = df_delta[df_delta['Significance'] == 'Yes']['Channel'].tolist()

# Print the list of significant channels
print("\nSignificant Channels:")
print(significant_channels)

DELTA: 
 T_stat:
 [ 0.40717437  3.75145542 -0.68082492  6.443173    5.74488966  4.30429819
  0.94458015 -2.20689472  5.0966785   3.81682569 -0.10960551 -0.81695783
  4.87370662  3.56629918  0.86168157 -1.55458046 -1.09745587  4.80153134
 -0.53682054  2.38517199  0.85983759  0.9854206   2.16113947  5.25840201
  3.17864823  2.82173405  3.66880429  1.2431775   3.94891583  4.47062735
  3.07666682  0.94437036  0.71504365  4.13051545  0.8439041   6.18236807
  4.85057084  2.99856227  1.83448017  5.59331717  3.30910621  1.02677543
  2.33386475  3.3335244   1.71149292  0.97175768  0.62539914  3.59262953
  3.63483232  0.31907784  0.74563494  0.19921202  2.03035844  1.1783698
  2.01150463  1.96373282  3.4740926   2.8504699   2.28475413  2.57674626
  1.46483013] 
 P_stat:
 [6.84107340e-01 2.09474171e-04 4.96412025e-01 4.89500511e-10
 2.39881790e-08 2.16003662e-05 3.45460884e-01 2.82364412e-02
 5.90038925e-07 1.60050585e-04 9.12775437e-01 4.14420103e-01
 1.79478975e-06 4.14365814e-04 3.89607984e-01

In [22]:
df_delta.to_excel('t-test_delta.xlsx')
df_delta

,Channel,T_Stat,P_Value,Significance
0,Fp1,0.407174,6.841073e-01,No
1,AF3,3.751455,2.094742e-04,Yes
2,AF7,-0.680825,4.964120e-01,No
3,Fz,6.443173,4.895005e-10,Yes
4,F1,5.744890,2.398818e-08,Yes
...,...,...,...,...
56,P8,3.474093,5.685736e-04,Yes
57,POz,2.850470,4.596252e-03,Yes
58,PO4,2.284754,2.283538e-02,Yes
59,PO8,2.576746,1.031644e-02,Yes


In [23]:
t_stat_theta, p_theta_t = stats.ttest_ind(theta_power_SD, theta_power_NS, equal_var=False)
z_stat_theta, p_theta_z = ztest(theta_power_SD, theta_power_NS, value=0)
print("THETA: \n","T_stat:\n", t_stat_theta,"\n","P_stat:\n", p_theta_t,"\n Z_stat:\n", z_stat_theta,"\n p_stat:\n", p_theta_z)

eeg_channels = ["Fp1", "AF3", "AF7", "Fz", "F1", "F3", "F5", "F7","FC1", "FC3", "FC5", "FT7","Cz", "C1", "C3", "C5", "T7","CP1", "CP3", "CP5", "TP7", "TP9","Pz", "P1", "P3", "P5", "P7","PO3", "PO7", "Oz", "O1","Fpz", "Fp2", "AF4", "AF8","F2", "F4", "F6", "F8","FC2", "FC4", "FC6", "FT8","C2", "C4", "C6", "T8","CPz", "CP2", "CP4", "CP6","TP8", "TP10","P2", "P4", "P6", "P8","POz", "PO4", "PO8", "O2"]
# Create the DataFrame
df_theta = pd.DataFrame({
    'Channel': eeg_channels,
    'T_Stat': t_stat_theta,
    'P_Value': p_theta_t,
    'Significance': np.where(p_theta_t < 0.05, 'Yes', 'No') 
})
print(df_theta['Significance'].value_counts())
# Filter the DataFrame for significant channels
significant_channels = df_theta[df_theta['Significance'] == 'Yes']['Channel'].tolist()

# Print the list of significant channels
print("\nSignificant Channels:")
print(significant_channels)

THETA: 
 T_stat:
 [ 2.07660534  3.31515867  1.32362507  3.63519231  3.90831332  3.16327114
  2.35019376  0.90633715  4.40885856  2.77612717  1.04152987  0.44050497
  4.93661376  2.99072564  0.29006546  0.38942561  0.9006811   1.79045504
 -0.09889851  0.93036353  2.04420358  4.40448152  2.21910853  2.6743503
  1.74528174  1.68927181  2.29637164  1.44219466  3.21146243  4.17280843
  3.91711434  2.66227317  2.70046903  2.761717    3.52917836  4.4452439
  3.35350137  3.16650721  3.007419    4.36758198  2.71053914  2.23316711
  3.68290663  3.19425311  0.84105898  1.50691007  3.47978949  2.86972174
  2.06141906  0.59391131  1.13014598  2.39972176  3.69676405  0.22347535
  0.58883442  0.39435205  2.33454193  1.39664657  0.59681058  2.01358779
  1.01349287] 
 P_stat:
 [3.84497060e-02 1.02577008e-03 1.86351631e-01 3.30954895e-04
 1.14123032e-04 1.68787135e-03 1.92512542e-02 3.65313227e-01
 1.33899427e-05 5.75277094e-03 2.98262514e-01 6.59807116e-01
 1.20822453e-06 2.95090046e-03 7.71910672e-01 

In [24]:
df_theta.to_excel('t-test_theta.xlsx')
df_theta

,Channel,T_Stat,P_Value,Significance
0,Fp1,2.076605,0.038450,Yes
1,AF3,3.315159,0.001026,Yes
2,AF7,1.323625,0.186352,No
3,Fz,3.635192,0.000331,Yes
4,F1,3.908313,0.000114,Yes
...,...,...,...,...
56,P8,2.334542,0.020042,Yes
57,POz,1.396647,0.163263,No
58,PO4,0.596811,0.550957,No
59,PO8,2.013588,0.044694,Yes


In [25]:
t_stat_alpha, p_alpha_t = stats.ttest_ind(alpha_power_SD, alpha_power_NS, equal_var=False)
z_stat_alpha, p_alpha_z = ztest(alpha_power_SD, alpha_power_NS, value=0)
print("ALPHA: \n","T_stat:\n", t_stat_alpha,"\n","P_stat:\n", p_alpha_t,"\n Z_stat:\n", z_stat_alpha,"\n p_stat:\n", p_alpha_z)
# Create the DataFrame
df_alpha = pd.DataFrame({
    'Channel': eeg_channels,
    'T_Stat': t_stat_alpha,
    'P_Value': p_alpha_t,
    'Significance': np.where(p_alpha_t < 0.05, 'Yes', 'No') 
})
print(df_alpha['Significance'].value_counts())
# Filter the DataFrame for significant channels
significant_channels = df_alpha[df_alpha['Significance'] == 'Yes']['Channel'].tolist()

# Print the list of significant channels
print("\nSignificant Channels:")
print(significant_channels)

ALPHA: 
 T_stat:
 [-0.89087872 -1.24008569 -1.02150847 -1.48808433 -1.43540933 -1.61922721
 -1.18570231 -1.86632461 -1.9848942  -2.30273753 -2.19329417 -1.70184522
 -1.96760191 -3.15524172 -3.87507898 -3.05641327 -1.71560787 -1.52497848
 -2.68087602 -2.56362208 -0.65630354  0.80525034 -1.85046666 -0.28260537
 -1.48492354 -2.0119743  -1.49486286 -1.88809099 -1.26712887 -0.12376553
 -0.54335923 -0.93669295 -1.04386069 -1.33098784 -1.15069463 -1.61297443
 -1.83921831 -1.54373247 -1.67915196 -2.59192406 -2.70200289 -1.46070707
 -0.07701017 -2.82943326 -2.22903556 -0.70810668  1.33472069 -1.54636376
 -1.54502327 -2.14652175 -2.00497138 -1.2156697   0.65323956 -2.25878443
 -2.01964987 -2.06360698 -1.21190331 -1.97139254 -2.08860731 -1.67429017
 -1.47139701] 
 P_stat:
 [3.73667757e-01 2.15976784e-01 3.07816730e-01 1.37914901e-01
 1.52341692e-01 1.06559474e-01 2.36771905e-01 6.29852276e-02
 4.82201120e-02 2.20583428e-02 2.91126420e-02 8.96907936e-02
 5.01510305e-02 1.77986662e-03 1.30851869e-0

In [26]:
df_alpha.to_excel('t-test_alpha.xlsx')
df_alpha

,Channel,T_Stat,P_Value,Significance
0,Fp1,-0.890879,0.373668,No
1,AF3,-1.240086,0.215977,No
2,AF7,-1.021508,0.307817,No
3,Fz,-1.488084,0.137915,No
4,F1,-1.435409,0.152342,No
...,...,...,...,...
56,P8,-1.211903,0.226646,No
57,POz,-1.971393,0.049614,Yes
58,PO4,-2.088607,0.037747,Yes
59,PO8,-1.674290,0.095254,No


In [27]:
t_stat_beta, p_beta_t = stats.ttest_ind(beta_power_SD, beta_power_NS, equal_var=False)
z_stat_beta, p_beta_z = ztest(beta_power_SD, beta_power_NS, value=0)
print("BETA: \n","T_stat:\n", t_stat_beta,"\n","P_stat:\n", p_beta_t,"\n Z_stat:\n", z_stat_beta,"\n p_stat:\n", p_beta_z)
# Create the DataFrame
df_beta = pd.DataFrame({
    'Channel': eeg_channels,
    'T_Stat': t_stat_beta,
    'P_Value': p_beta_t,
    'Significance': np.where(p_beta_t < 0.05, 'Yes', 'No') 
})
print(df_beta['Significance'].value_counts())

BETA: 
 T_stat:
 [ 4.23921669  2.51761494  2.89461319  3.48788287  4.19638024  2.10685905
  3.6771072  -1.63800711  2.4346715   0.38439848  0.43718471 -0.75323952
  2.37234663  0.71981584 -1.29217127  0.55822172 -2.09657848  0.40343489
 -0.91181191  0.68379487  1.56652715 -1.07968836  0.03696587  1.78299044
  0.21006788  0.10425453 -0.39350503 -1.28567865  1.79269481  2.66962433
  0.80086037  2.69775989  2.9439016   1.84958624  0.01882383  3.86027683
  1.74907517  1.45323086 -0.15063213  2.88356939  2.07821513  1.20372793
  0.34844189  1.56011634  0.6259856   0.53534189  0.4331205  -0.10863737
  0.39695496 -0.39493032  0.03435887  0.61240489  0.64550503 -1.48317219
 -0.51081074  0.18449754  1.62409546  0.80921695 -1.0936407   1.94756246
  0.27309647] 
 P_stat:
 [2.98602448e-05 1.24240455e-02 4.01084394e-03 5.41932115e-04
 3.39747349e-05 3.57260508e-02 2.76662603e-04 1.02214342e-01
 1.53424851e-02 7.00909085e-01 6.62216134e-01 4.51739368e-01
 1.81382014e-02 4.72041332e-01 1.97016838e-01

In [28]:
df_beta.to_excel('t-test_beta.xlsx')
df_beta

,Channel,T_Stat,P_Value,Significance
0,Fp1,4.239217,0.000030,Yes
1,AF3,2.517615,0.012424,Yes
2,AF7,2.894613,0.004011,Yes
3,Fz,3.487883,0.000542,Yes
4,F1,4.196380,0.000034,Yes
...,...,...,...,...
56,P8,1.624095,0.105304,No
57,POz,0.809217,0.418857,No
58,PO4,-1.093641,0.274829,No
59,PO8,1.947562,0.052687,No


In [29]:
t_stat_gamma, p_gamma_t = stats.ttest_ind(gamma_power_SD, gamma_power_NS, equal_var=False)
z_stat_gamma, p_gamma_z = ztest(gamma_power_SD, gamma_power_NS, value=0)
print("GAMMA: \n","T_stat:\n", t_stat_gamma,"\n","P_stat:\n", p_gamma_t,"\n Z_stat:\n", z_stat_gamma,"\n p_stat:\n", p_gamma_z)
# Create the DataFrame
df_gamma = pd.DataFrame({
    'Channel': eeg_channels,
    'T_Stat': t_stat_gamma,
    'P_Value': p_beta_t,
    'Significance': np.where(p_gamma_t < 0.05, 'Yes', 'No') 
})
print(df_gamma['Significance'].value_counts())

GAMMA: 
 T_stat:
 [ 3.61259616  2.50082125  2.86454504  2.14468345  3.37987414  1.26632739
  3.28344294 -1.50793035  0.38182753 -0.3629625   0.50684319 -0.58817908
  0.38040204 -0.68871509 -1.13787021  0.638672   -1.66624988 -0.11553473
 -0.79620987  0.29171264  1.97160353  0.00569144 -0.52311298  0.694154
 -0.3501263   0.54703573 -0.18820747 -1.13781103  1.78516385  2.18648556
  0.69962977  3.0580658   2.97910934  1.12452622 -0.30384768  3.56797712
  0.73682723  1.60793633 -0.81706975  2.16545243  1.40472528  0.31701325
 -0.7138247   1.5697116  -0.58409226 -0.16874543 -0.20608572 -1.29444322
 -0.77333442 -1.15566036 -0.42689489 -0.02730465 -0.24574671 -2.00681384
 -1.1325491   1.27875162  1.80110983  0.89103337 -1.61510414  1.75155368
 -0.26295061] 
 P_stat:
 [3.61274844e-04 1.29462282e-02 4.39294600e-03 3.25544391e-02
 7.95879227e-04 2.06158012e-01 1.12184767e-03 1.32360777e-01
 7.02809854e-01 7.16892209e-01 6.12578691e-01 5.56730813e-01
 7.03867256e-01 4.91535391e-01 2.56010595e-01 

In [30]:
df_gamma.to_excel('t-test_gamma.xlsx')
df_gamma

,Channel,T_Stat,P_Value,Significance
0,Fp1,3.612596,0.000030,Yes
1,AF3,2.500821,0.012424,Yes
2,AF7,2.864545,0.004011,Yes
3,Fz,2.144683,0.000542,Yes
4,F1,3.379874,0.000034,Yes
...,...,...,...,...
56,P8,1.801110,0.105304,No
57,POz,0.891033,0.418857,No
58,PO4,-1.615104,0.274829,No
59,PO8,1.751554,0.052687,No
